In [8]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [14]:
from anthropic.types import Message

# Helper functions
def add_user_message(messages, message):
    user_message = { 
        "role": "user", 
        "content": message.content if isinstance(message, Message) else message
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant", 
        "content": message.content if isinstance(message, Message) else message}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [19]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

# batch_tool_schema = {
#     "name": "batch_tool",
#     "description": "Invoke multiple other tool calls simultaneously",
#     "input_schema": {
#         "type": "object",
#         "properties": {
#             "invocations": {
#                 "type": "array",
#                 "description": "The tool calls to invoke",
#                 "items": {
#                     "type": "object",
#                     "properties": {
#                         "name": {
#                             "type": "string",
#                             "description": "The name of the tool to invoke",
#                         },
#                         "arguments": {
#                             "type": "string",
#                             "description": "The arguments to the tool, encoded as a JSON string",
#                         },
#                     },
#                     "required": ["name", "arguments"],
#                 },
#             }
#         },
#         "required": ["invocations"],
#     },
# }

pass

In [20]:
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format can not be empty")
    
    return datetime.now().strftime(date_format)

# get_current_datetime("%H:%M")
# get_current_datetime("")

get_current_datetime_schema = ToolParam({
    "name" : "get_current_datetime",
    "description": "Returns the current local date and time as a formatted string. Use this tool whenever the user asks for the current time, today's date, or a timestamp. The output format is controlled by a strftime-compatible format string. If no format is provided, the default ISO-like format '%Y-%m-%d %H:%M:%S' is used (e.g. '2025-03-14 09:05:30').",
     "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls the structure of the returned datetime. Must not be an empty string. Defaults to '%Y-%m-%d %H:%M:%S'. Common examples: '%Y-%m-%d' for date only (e.g. '2025-03-14'), '%H:%M:%S' for time only (e.g. '09:05:30'), '%d/%m/%Y' for European date style (e.g. '14/03/2025'), '%B %d, %Y' for long-form date (e.g. 'March 14, 2025'), '%I:%M %p' for 12-hour clock (e.g. '09:05 AM').",
        "default": "%Y-%m-%d %H:%M:%S",
        "minLength": 1
      }
    },
    "required": []
  } 
})
    

In [6]:
messages = []
messages.append(
    {
        "role" : "user",
        "content" : "What is the exact time , formatted as HH:MM:SS",
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({
    "role" : "assistant",
    "content" : response.content,
})

messages

[{'role': 'user', 'content': 'What is the exact time , formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01E6fBggCAZGETf8dbHrCoPR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [23]:
# response
# response.content[0].input
result = get_current_datetime(**response.content[0].input)

In [24]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": response.content[0].id,
                "content": result,
                "is_error": False
            }
        ]
    }
)

messages

[{'role': 'user', 'content': 'What is the exact time , formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01E6fBggCAZGETf8dbHrCoPR', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01E6fBggCAZGETf8dbHrCoPR',
    'content': '17:40:33',
    'is_error': False}]}]

In [ ]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

In [26]:
response

Message(id='msg_011CcrW4CX9WCQMWrBHJdGuQ', container=None, content=[TextBlock(citations=None, text='The exact time is **17:40:33** (5:40:33 PM in 12-hour format).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=932, output_tokens=29, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [21]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [ ]:
def run_conversation(messages):
    while True:
        response = chat(messages, 
                        tools=[ get_current_datetime_schema, 
                                add_duration_to_datetime_schema,
                                set_reminder_schema,])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages

In [23]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.",
)
run_conversation(messages)


I can help you calculate when your doctor's appointment is! Let me figure out the date that is 177 days after January 1st, 2050.

January 1st, 2050 + 177 days = **June 27th, 2050**

However, I don't have the ability to actually set reminders on your device or in calendar applications. To set a reminder for your doctor's appointment on **June 27, 2050**, you can:

1. **Use your phone's calendar app** - Create an event on June 27, 2050 and set a reminder notification
2. **Use your computer's calendar** - Add it to Google Calendar, Outlook, Apple Calendar, etc.
3. **Use a voice assistant** - Say "Set a reminder for my doctor's appointment on June 27, 2050"
4. **Use a notes app** - Write it down with a reminder notification if your notes app supports it

Would you like me to help with anything else related to this appointment?


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01Wxn2k1Jz8eB7CvNJjxG3dZ', caller=DirectCaller(type='direct'), input={}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Wxn2k1Jz8eB7CvNJjxG3dZ',
    'content': '"2026-07-19 20:21:04"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='I can help you calculate when your doctor\'s appointment is! Let me figure out the date that is 177 days after January 1st, 2050.\n\nJanuary 1st, 2050 + 177 days = **June 27th, 2050**\n\nHowever, I don\'t have the ability to actually set reminders on your device or in calendar applications. To set a reminder for your doctor\'s appointment on **June 27, 2050**, you can:\n\n1. **Use your phone\'s calendar app** - Create an event on June 27, 2050 and set a remi